# 📊 Gemini Stateless ASR Context Window Benchmarking Sweep

This notebook is an interactive, cloud-native research dashboard designed to evaluate ASR accuracy (WER/CER) and turn latency across different context window sizes (`[1, 5, 11, 21, 31]`).

It implements a clean, unpolluted, and deterministic evaluation sweep:
1. **Symmetric Sampling:** Deterministically samples a consistent set of deep recording channels from GCS.
2. **Stateless Execution:** Invokes our high-performance **Audio-Only Stateless** engine (`gemini.stateless`) to transcribe segments concurrently.
3. **Statistical Evaluation:** Computes WER/CER on-the-fly and renders publication-grade charts with 95% Confidence Interval error bands to visualize the Accuracy vs. Latency trade-offs.


In [ ]:
# @title Bootstrap and Install Environment
import os

# Clone repository if not already present (for hosted Colab environments)
if not os.path.exists("radio-transcription"):
    !git clone -q https://github.com/watch-duty/radio-transcription.git

# Install the model library in editable mode along with required dependencies
try:
    import common

    print("✅ Library 'common' already installed.")
except ImportError:
    print("Installing library and dependencies...")
    %pip install -q -e radio-transcription/model loguru tqdm jiwer

    import site
    import importlib

    importlib.reload(site)
    print("\n✅ Dependencies installed successfully.")

In [ ]:
# @title Imports
import asyncio
import collections
import json
import os
import re
import sys
import time
from collections import defaultdict, deque
from pathlib import Path
from typing import Any

from google.cloud import storage
from google.colab import auth, userdata
from IPython.display import display
from loguru import logger
from tqdm.asyncio import tqdm
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import jiwer

# GenAI imports
from google.genai import types, Client as GenAiClient

In [ ]:
# @title Authenticate with GCP
# @markdown Run this cell to authenticate your browser session with Google Cloud.
print("Attempting standard browser authentication...")
auth.authenticate_user()
print("✅ Browser session authenticated successfully.")

In [ ]:
# @title Benchmarking Configuration

# @markdown ### Model Selection
MODEL_ID = "gemini-3.1-flash-lite"  # @param ["gemini-3.1-flash-lite", "gemini-3-flash-preview", "gemini-3.1-pro-preview", "gemini-3.5-flash"] {type:"string"}
GCP_PROJECT_ID = userdata.get("GCP_PROJECT_ID")
GCS_BUCKET = userdata.get("GCS_BUCKET")

# @markdown ### GCP Infrastructure Configuration
GCP_LOCATION = "us"  # @param {type:"string"}

# @markdown ### Input Manifest Configuration
# @markdown Partial path under `gs://{GCS_BUCKET}/segmented_audio/` where `batch_manifest.jsonl` is located.
INPUT_AUDIO_DIR = "broadcastify/calls/eval_audio_masked_v2"  # @param {type:"string"}  # fmt: skip

# @markdown ### Concurrency Settings
CONCURRENCY_LIMIT = 5  # @param {type:"integer"}

assert INPUT_AUDIO_DIR, "INPUT_AUDIO_DIR must be provided and cannot be empty."
assert GCP_PROJECT_ID, "GCP_PROJECT_ID must be provided in Colab userdata."
assert GCS_BUCKET, "GCS_BUCKET must be provided in Colab userdata."

MODEL_PATH = f"projects/{GCP_PROJECT_ID}/locations/{GCP_LOCATION}/publishers/google/models/{MODEL_ID}"
MANIFEST_URI = (
    f"gs://{GCS_BUCKET}/segmented_audio/{INPUT_AUDIO_DIR}/batch_manifest.jsonl"
)

SYSTEM_PROMPT = """\
Your primary task is to produce a strict, verbatim transcription of the spoken audio. Your absolute highest priority is to transcribe only what you hear with high acoustic certainty. Do not add, invent, or infer any speech that is not clearly audible. The audio may originate from VHF/UHF radio traffic and can include mic clicks, RF static, radio hum, and potentially unintelligible speech. When the audio is unequivocally confirmed as fire-related dispatch, speakers often use heavy jargon, and specific formatting rules apply.

EXPECTED TERMINOLOGY:
These are terms and unit identifiers commonly used in fire-related dispatch. These terms and formatting rules apply exclusively to audio that is unequivocally confirmed as fire-related dispatch. If these exact terms are clearly heard in the audio, transcribe them as listed. Do not invent or infer the use of these terms if they are not genuinely spoken.
copy, received, affirmative, affirm, proceed, responding, responding to, en-route, on-scene, in the area, available, returning, in service, got a caller, caller advising, in quarters, arrived, go ahead, back at, engine, tanker, brush, brush truck, tender, battalion, squad, ladder, tower, tower-ladder, medic, ambulance, k, branch, chopper, copter, AIQ, AOR, DO, IC, ICP, LAT, RP, SEAT, TAC, VFIRE, VLAT, patrol, rescue, station, personnel, air attack, air tactics, helispot, lead plane, strike team, control, being toned, box alarm, cancel the balance, chaparral, exposure protection, fire attack, fire boss, forward progress stopped, forward rate of spread stopped, heavy timber, left flank, light flashy fuels, rate of spread, right flank, structure defense, structure protection, structures threatened, terrain driven, wind driven, clear, clear and in service, code 1, code 2, code 3, code 4, code 33, medical call, fire alarm, commercial fire alarm, breathing problem, cardiac, heart problem, diabetic shock, mvc, trespass, harassment, 10-4, 10-7, 10-8, 10-9, 10-15, 10-20, 10-22, 10-23, 10-91, 10-97.

CRITICAL RULES:
1. Output the transcript strictly and precisely as spoken in the audio, with no newlines. Do not add, invent, or infer any speech that is not clearly audible.
2. When transcribing numbers, write the digits grouped together (e.g., 100, 6333).
3. If the audio contains a unit identifier, format it as the unit type followed by digits (e.g., Engine 41, Battalion 2). Apply this rule strictly only if the unit identifier is clearly spoken AND the context is unequivocally fire-related dispatch.
4. Transcribe only the duration of speech present. Do not extend the transcription with additional words or phrases that were not spoken, even if contextually plausible.

QUALITY GATE: Your absolute highest priority is to transcribe only what you hear with high acoustic certainty.
    *   If the audio contains clear speech that is not fire-related dispatch, you MUST transcribe it verbatim, exactly as heard, without applying any fire-specific formatting or jargon, and without attempting to interpret it as fire dispatch traffic.
    *   If a portion of audio is obscured, noisy, ambiguous, or contains speech that cannot be confidently identified, you MUST replace that specific portion with [UNINTELLIGIBLE].
    *   Do not attempt to infer, guess, or invent speech to fit any expected context or terminology list.
    *   Do not attempt to phonetically guess ambiguous noise.
    *   If the entire audio segment does not contain any discernible speech, output only [UNINTELLIGIBLE].

TASK:
You will receive a sequence of audio files representing the history of the channel. The final audio file in the list is the new segment. Use the preceding audio files ONLY as acoustic and vocabulary context reference. Transcribe ONLY the final audio file verbatim. Output strictly the transcript of the final segment.
"""

SAFETY_SETTINGS = [
    {"category": "HARM_CATEGORY_HATE_SPEECH", "threshold": "BLOCK_NONE"},
    {"category": "HARM_CATEGORY_SEXUALLY_EXPLICIT", "threshold": "BLOCK_NONE"},
    {"category": "HARM_CATEGORY_DANGEROUS_CONTENT", "threshold": "BLOCK_NONE"},
    {"category": "HARM_CATEGORY_HARASSMENT", "threshold": "BLOCK_NONE"},
]

GENERATION_CONFIG = {
    "temperature": 0.0,
    "max_output_tokens": 512,
}

logger.remove()
_ = logger.add(
    sys.stderr, format="<level>{level}</level>: {message}", level="INFO"
)

In [ ]:
# @title Run Stateless Parameter Sweep & Plot Curves

# Simple text normalizer for ASR cleaning in Colab (numbers, punctuation, spacing)
def clean_text_for_wer(text: str) -> str:
    if not text:
        return ""
    text = text.lower().strip()
    text = re.sub(
        r"[^\w\s\-\:]", "", text
    )  # Remove punctuation except hyphens/colons
    text = re.sub(r"\s+", " ", text)  # Collapse spacing
    return text.strip()


# Async Retry Policy protecting GCS streaming network channels
custom_async_retry = api_retry_async.AsyncRetry(
    predicate=api_retry.if_exception_type(
        (ClientError, GoogleAPICallError, asyncio.TimeoutError, ConnectionError)
    ),
    initial=1.0,
    maximum=30.0,
    multiplier=2.0,
    deadline=120.0,
)


# Core stateless transcription loop (reused locally for the sweep)
async def process_single_channel_stateless_eval(
    channel_id: str,
    segments: list[dict],
    genai_client: GenAiClient,
    model_path: str,
    system_prompt: str,
    safety_settings: list,
    generation_config: dict,
    context_size: int,
    semaphore: asyncio.Semaphore,
) -> list[dict]:
    results = []
    max_history_turns = max(0, context_size - 1)
    history_buffer = deque(maxlen=max_history_turns)

    async with semaphore:
        for entry in segments:
            uri = entry["audio_filepath"]
            reference = (
                entry.get("text")
                or entry.get("transcript")
                or entry.get("reference")
                or ""
            )

            parts = []
            if len(history_buffer) > 0:
                parts.append(
                    types.Part.from_text(
                        text="--- HISTORICAL AUDIO CONTEXT ---"
                    )
                )
                for past_audio_uri in history_buffer:
                    parts.append(
                        types.Part.from_uri(
                            file_uri=past_audio_uri, mime_type="audio/flac"
                        )
                    )
            parts.append(
                types.Part.from_text(text="--- NEW TARGET AUDIO SEGMENT ---")
            )
            parts.append(
                types.Part.from_uri(file_uri=uri, mime_type="audio/flac")
            )

            contents = [types.Content(role="user", parts=parts)]

            transcript = None
            error_msg = None
            req_start_time = time.time()

            @custom_async_retry
            async def execute_call():
                return await genai_client.aio.models.generate_content(
                    model=model_path,
                    contents=contents,
                    config=types.GenerateContentConfig(
                        system_instruction=system_prompt.strip(),
                        safety_settings=safety_settings,
                        temperature=generation_config["temperature"],
                        max_output_tokens=generation_config[
                            "max_output_tokens"
                        ],
                        thinking_config=types.ThinkingConfig(thinking_budget=0),
                    ),
                )

            try:
                response = await asyncio.wait_for(execute_call(), timeout=60.0)
                if response.candidates and response.candidates[0].content.parts:
                    transcript = response.text.strip()
                else:
                    error_msg = "Empty response."
            except Exception as e:
                # Define is_fatal_error helper locally to support fail-fast
                def is_fatal_error(exc: Exception) -> bool:
                    exc_str = str(exc).upper()
                    for fatal_keyword in (
                        "400",
                        "403",
                        "404",
                        "PERMISSION_DENIED",
                        "NOT_FOUND",
                        "INVALID_ARGUMENT",
                    ):
                        if fatal_keyword in exc_str:
                            return True
                    return False

                if is_fatal_error(e):
                    print(
                        f"FATAL API ERROR for segment: {Path(uri).name} | Error: {e!s}. Aborting sweep."
                    )
                    raise
                error_msg = str(e)

            latency = time.time() - req_start_time

            # Calculate WER/CER
            ref_clean = clean_text_for_wer(reference)
            hyp_clean = clean_text_for_wer(transcript)

            seg_wer = 1.0
            seg_cer = 1.0
            if ref_clean and transcript is not None:
                try:
                    seg_wer = jiwer.wer(ref_clean, hyp_clean)
                    seg_cer = jiwer.cer(ref_clean, hyp_clean)
                except Exception:
                    pass
            elif not ref_clean and not hyp_clean and transcript is not None:
                seg_wer = 0.0
                seg_cer = 0.0

            results.append(
                {
                    "context_size": context_size,
                    "channel_id": channel_id,
                    "audio_filepath": uri,
                    "reference": reference,
                    "hypothesis": transcript or "",
                    "latency": latency,
                    "wer": seg_wer,
                    "cer": seg_cer,
                    "error": error_msg,
                }
            )

            history_buffer.append(uri)

    return results


async def run_stateless_benchmarking_sweep():
    # 1. Initialize Clients
    print("Initializing Google Cloud and GenAI clients...")
    if GCP_LOCATION == "us":
        base_url = "https://aiplatform.us.rep.googleapis.com"
    else:
        base_url = (
            f"https://{GCP_LOCATION}-aiplatform.googleapis.com"
            if GCP_LOCATION and GCP_LOCATION != "global"
            else "https://aiplatform.googleapis.com"
        )
    genai_client = GenAiClient(
        project=GCP_PROJECT_ID,
        location=GCP_LOCATION,
        vertexai=True,
        http_options=types.HttpOptions(base_url=base_url),
    )
    storage_client = storage.Client(project=GCP_PROJECT_ID)

    # 2. Load Manifest and Group by Channel
    print("📢 Loading manifest from GCS...")
    m_bucket = MANIFEST_URI.replace("gs://", "").split("/")[0]
    m_path = "/".join(MANIFEST_URI.replace("gs://", "").split("/")[1:])
    manifest_blob = storage_client.bucket(m_bucket).blob(m_path)

    if not manifest_blob.exists():
        raise FileNotFoundError(
            f"Manifest GCS file not found at {MANIFEST_URI}"
        )

    content = manifest_blob.download_as_text().strip().split("\n")
    channels = defaultdict(list)

    for line in content:
        if line.strip():
            entry = json.loads(line)
            parent_audio_unit = Path(entry["audio_filepath"]).parent.name
            entry["example_id"] = parent_audio_unit
            channels[parent_audio_unit].append(entry)

    for ch in channels:
        channels[ch].sort(
            key=lambda x: (
                x.get("offset", 0),
                x.get("start_time", 0),
                x.get("audio_filepath", ""),
            )
        )

    # Sweep configurations
    context_sizes = [1, 5, 11, 21, 31]
    max_size = max(context_sizes)

    print(f"Filtering for channels with at least {max_size} segments...")
    qualified_channels = {
        cid: entries
        for cid, entries in channels.items()
        if len(entries) >= max_size
    }

    if not qualified_channels:
        raise RuntimeError(
            f"No channels found in the manifest with at least {max_size} segments!"
        )

    # Deterministically sample 3 channels
    random.seed(42)
    sampled_cids = sorted(list(qualified_channels.keys()))
    sampled_cids = random.sample(sampled_cids, min(3, len(sampled_cids)))

    print(
        f"✅ Sampleed {len(sampled_cids)} channels for evaluation (Seed: 42):"
    )
    for cid in sampled_cids:
        print(f"   - {cid}: {len(qualified_channels[cid])} segments")

    raw_results = []
    semaphore = asyncio.Semaphore(CONCURRENCY_LIMIT)

    # 3. Run the Sweep
    for size in context_sizes:
        print("\n" + "=" * 80)
        print(
            f"🚀 RUNNING SWEEP FOR CONTEXT WINDOW SIZE: {size} (NUM_RECENT_EVENTS)"
        )
        print("=" * 80)

        async def evaluate_channel(cid, entries):
            eval_entries = entries[:max_size]  # Keep it fast and unpolluted
            return await process_single_channel_stateless_eval(
                channel_id=cid,
                segments=eval_entries,
                genai_client=genai_client,
                model_path=MODEL_PATH,
                system_prompt=SYSTEM_PROMPT,
                safety_settings=SAFETY_SETTINGS,
                generation_config=GENERATION_CONFIG,
                context_size=size,
                semaphore=semaphore,
            )

        loop = asyncio.get_running_loop()
        tasks = [
            loop.create_task(evaluate_channel(cid, qualified_channels[cid]))
            for cid in sampled_cids
        ]
        try:
            channel_results = await asyncio.gather(*tasks)
        except Exception as e:
            print(
                f"FATAL Sweep Failure detected: {e!s}. Cancelling all other active channels..."
            )
            for t in tasks:
                if not t.done():
                    t.cancel()
            await asyncio.gather(*tasks, return_exceptions=True)
            raise

        size_results = [item for sublist in channel_results for item in sublist]
        raw_results.extend(size_results)

        # Aggregates
        df_size = pd.DataFrame(size_results)
        successful_df = df_size[
            df_size["error"].isna() | (df_size["error"] == "")
        ]

        avg_wer = (
            successful_df["wer"].mean() * 100
            if not successful_df.empty
            else 0.0
        )
        avg_cer = (
            successful_df["cer"].mean() * 100
            if not successful_df.empty
            else 0.0
        )
        avg_lat = (
            successful_df["latency"].mean() if not successful_df.empty else 0.0
        )

        print(f"📊 SUMMARY FOR SIZE {size}:")
        print(f"   - Average WER: {avg_wer:.2f}% | Average CER: {avg_cer:.2f}%")
        print(f"   - Average Latency: {avg_lat:.2f} seconds")

    # 4. Save results
    df_raw = pd.DataFrame(raw_results)
    df_raw.to_csv("stateless_sweep_results.csv", index=False)
    print("\n💾 Raw results saved to 'stateless_sweep_results.csv'!")

    # 5. Generate Stats & Plots
    print("\n📈 Plotting curves...")
    summary_data = []
    for size in context_sizes:
        df_sub = df_raw[df_raw["context_size"] == size]
        successful_sub = df_sub[
            df_sub["error"].isna() | (df_sub["error"] == "")
        ]
        n_segments = len(successful_sub)

        mean_wer = successful_sub["wer"].mean() * 100 if n_segments > 0 else 0.0
        sem_wer = (successful_sub["wer"].sem() * 100) if n_segments > 1 else 0.0
        ci_wer = 1.96 * sem_wer

        mean_cer = successful_sub["cer"].mean() * 100 if n_segments > 0 else 0.0
        sem_cer = (successful_sub["cer"].sem() * 100) if n_segments > 1 else 0.0
        ci_cer = 1.96 * sem_cer

        mean_lat = successful_sub["latency"].mean() if n_segments > 0 else 0.0
        sem_lat = (successful_sub["latency"].sem()) if n_segments > 1 else 0.0
        ci_lat = 1.96 * sem_lat

        summary_data.append(
            {
                "Context Size": size,
                "WER (%)": f"{mean_wer:.2f}% ± {ci_wer:.2f}%"
                if n_segments > 0
                else "N/A",
                "CER (%)": f"{mean_cer:.2f}% ± {ci_cer:.2f}%"
                if n_segments > 0
                else "N/A",
                "Avg Latency (s)": f"{mean_lat:.2f}s ± {ci_lat:.2f}s"
                if n_segments > 0
                else "N/A",
                "raw_wer_mean": mean_wer,
                "raw_wer_ci": ci_wer,
                "raw_cer_mean": mean_cer,
                "raw_cer_ci": ci_cer,
                "raw_lat_mean": mean_lat,
                "raw_lat_ci": ci_lat,
            }
        )

    df_sum = pd.DataFrame(summary_data)
    display(df_sum[["Context Size", "WER (%)", "CER (%)", "Avg Latency (s)"]])

    # Render Plots
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.set_theme(style="whitegrid")

    sizes_numeric = [int(s) for s in context_sizes]

    # Plot 1: WER vs Context Size with CI bands
    axes[0].errorbar(
        sizes_numeric,
        df_sum["raw_wer_mean"],
        yerr=df_sum["raw_wer_ci"],
        fmt="-o",
        color="royalblue",
        ecolor="cornflowerblue",
        elinewidth=2,
        capsize=4,
        label="WER",
    )
    axes[0].set_title(
        "Word Error Rate (WER) vs. Lookback Size\n(Lower is Better)",
        fontsize=12,
        fontweight="bold",
    )
    axes[0].set_xlabel("Lookback Window (num_recent_events)", fontsize=10)
    axes[0].set_ylabel("WER (%)", fontsize=10)
    axes[0].set_xticks(sizes_numeric)

    # Plot 2: Latency vs Context Size
    axes[1].errorbar(
        sizes_numeric,
        df_sum["raw_lat_mean"],
        yerr=df_sum["raw_lat_ci"],
        fmt="-o",
        color="darkorange",
        ecolor="moccasin",
        elinewidth=2,
        capsize=4,
        label="Latency",
    )
    axes[1].set_title(
        "Average Turn Latency vs. Lookback Size\n(Lower is Better)",
        fontsize=12,
        fontweight="bold",
    )
    axes[1].set_xlabel("Lookback Window (num_recent_events)", fontsize=10)
    axes[1].set_ylabel("Latency (seconds)", fontsize=10)
    axes[1].set_xticks(sizes_numeric)

    plt.suptitle(
        "Gemini Stateless Audio-Only ASR Lookback Window Evaluation (1 - 31)",
        fontsize=14,
        fontweight="bold",
        y=1.02,
    )
    plt.tight_layout()
    plt.show()


await run_stateless_benchmarking_sweep()